# Final Project Pembelajaran Mesin (F) - Kelompok 5
- Mario Napitupulu (5025241085)
- Willy Marcelius (5025241096)
- Rendy Tanuwijaya (5025241099)
- Hajendra Herlambang (5025241101)

Notebook ini diawali dengan tahap preprocessing untuk dataset Ames House Prices. Fokus preprocessing yang digunakan:

1. Feature engineering
2. Normalisasi fitur numerik
3. Encoding fitur kategorikal dengan ordinal encoding dan one-hot encoding

## 1. Preprocessing

### Import Library dan Load Dataset

In [1]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

pd.set_option('display.max_columns', 120)

from pathlib import Path

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / 'data' if (PROJECT_DIR / 'data').exists() else PROJECT_DIR.parent / 'data'
train_df = pd.read_csv(DATA_DIR / 'train(1).csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')

print('Train shape:', train_df.shape)
print('Test shape :', test_df.shape)
train_df.head()

FileNotFoundError: [Errno 2] No such file or directory: '/data/train(1).csv'

### Identifikasi Missing Value

Pada dataset ini, beberapa nilai kosong muncul sebagai `NaN`. Untuk fitur seperti `Alley`, `PoolQC`, `Fence`, `FireplaceQu`, `Garage*`, dan `Bsmt*`, nilai kosong memiliki makna domain yaitu rumah tidak memiliki fasilitas tersebut. Karena itu, kolom-kolom tersebut akan diisi dengan kategori khusus `None`, bukan langsung dibuang.

In [ ]:
missing_summary = (
    train_df.isna().sum()
    .loc[lambda s: s > 0]
    .sort_values(ascending=False)
    .to_frame('jumlah_missing')
)
missing_summary['persentase'] = (missing_summary['jumlah_missing'] / len(train_df) * 100).round(2)
missing_summary

### Pemisahan Fitur dan Target

Kolom `SalePrice` digunakan sebagai target prediksi. Kolom `Id` tidak digunakan sebagai fitur karena hanya berfungsi sebagai identitas data.

In [ ]:
TARGET = 'SalePrice'
ID_COL = 'Id'

outlier_mask = (train_df['GrLivArea'] > 4000) & (train_df[TARGET] < 300000)
removed_outlier_ids = train_df.loc[outlier_mask, ID_COL].tolist()
model_train_df = train_df.loc[~outlier_mask].copy()

X = model_train_df.drop(columns=[TARGET, ID_COL])
y = model_train_df[TARGET]
X_test_final = test_df.drop(columns=[ID_COL])

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print('Outlier IDs removed:', removed_outlier_ids)
print('X_train:', X_train.shape)
print('X_valid:', X_valid.shape)
print('X_test_final:', X_test_final.shape)

### Feature Engineering Bersama

Feature engineering berikut digunakan bersama oleh seluruh model. Fitur yang ditambahkan mencakup total luas, total kamar mandi, umur bangunan, indikator fasilitas, interaksi kualitas dan luas, rasio, serta transformasi log untuk fitur numerik yang skewed.


In [ ]:
def safe_divide(numerator, denominator):
    return numerator / denominator.replace(0, np.nan)


ORDINAL_MAPS = {'ExterQual': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'], 'ExterCond': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'], 'BsmtQual': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'], 'BsmtCond': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'], 'HeatingQC': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'], 'KitchenQual': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'], 'FireplaceQu': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'], 'GarageQual': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'], 'GarageCond': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'], 'PoolQC': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'], 'BsmtExposure': ['None', 'No', 'Mn', 'Av', 'Gd'], 'BsmtFinType1': ['None', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'], 'BsmtFinType2': ['None', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'], 'Functional': ['None', 'Sal', 'Sev', 'Maj2', 'Maj1', 'Mod', 'Min2', 'Min1', 'Typ'], 'GarageFinish': ['None', 'Unf', 'RFn', 'Fin'], 'Fence': ['None', 'MnWw', 'GdWo', 'MnPrv', 'GdPrv'], 'LotShape': ['IR3', 'IR2', 'IR1', 'Reg'], 'LandSlope': ['Sev', 'Mod', 'Gtl'], 'PavedDrive': ['N', 'P', 'Y'], 'CentralAir': ['N', 'Y'], 'Street': ['Grvl', 'Pave']}

def engineer_features(df):
    df = df.copy()

    # This is a class label, not a continuous quantity.
    df["MSSubClass"] = df["MSSubClass"].astype(str)

    ordinal_scores = {
        col: {value: index for index, value in enumerate(values)}
        for col, values in ORDINAL_MAPS.items()
    }
    for col, mapping in ordinal_scores.items():
        if col in df.columns:
            df[f"{col}Score"] = df[col].fillna("None").map(mapping).fillna(-1)

    df["TotalSF"] = (
        df["TotalBsmtSF"].fillna(0)
        + df["1stFlrSF"].fillna(0)
        + df["2ndFlrSF"].fillna(0)
    )
    df["TotalBath"] = (
        df["FullBath"].fillna(0)
        + 0.5 * df["HalfBath"].fillna(0)
        + df["BsmtFullBath"].fillna(0)
        + 0.5 * df["BsmtHalfBath"].fillna(0)
    )
    df["HouseAge"] = (df["YrSold"] - df["YearBuilt"]).clip(lower=0)
    df["RemodAge"] = (df["YrSold"] - df["YearRemodAdd"]).clip(lower=0)
    df["GarageAge"] = (df["YrSold"] - df["GarageYrBlt"]).clip(lower=0)
    df["EffectiveAge"] = df[["HouseAge", "RemodAge"]].min(axis=1)
    df["TotalPorchSF"] = df[
        ["OpenPorchSF", "EnclosedPorch", "3SsnPorch", "ScreenPorch"]
    ].fillna(0).sum(axis=1)
    df["HasPool"] = (df["PoolArea"].fillna(0) > 0).astype(int)
    df["HasGarage"] = (df["GarageArea"].fillna(0) > 0).astype(int)
    df["HasBasement"] = (df["TotalBsmtSF"].fillna(0) > 0).astype(int)
    df["HasFireplace"] = (df["Fireplaces"].fillna(0) > 0).astype(int)
    df["TotalFinishedSF"] = (
        df["1stFlrSF"].fillna(0)
        + df["2ndFlrSF"].fillna(0)
        + df["BsmtFinSF1"].fillna(0)
        + df["BsmtFinSF2"].fillna(0)
    )
    df["TotalOutdoorSF"] = (
        df["WoodDeckSF"].fillna(0) + df["TotalPorchSF"] + df["PoolArea"].fillna(0)
    )
    df["QualSF"] = df["OverallQual"].fillna(0) * df["TotalSF"]
    df["QualBath"] = df["OverallQual"].fillna(0) * df["TotalBath"]
    df["GarageCarsArea"] = df["GarageCars"].fillna(0) * df["GarageArea"].fillna(0)
    df["Remodeled"] = (df["YearRemodAdd"] != df["YearBuilt"]).astype(int)
    df["NewerDwelling"] = (df["YearBuilt"] >= 2000).astype(int)
    df["Has2ndFloor"] = (df["2ndFlrSF"].fillna(0) > 0).astype(int)
    df["HasWoodDeck"] = (df["WoodDeckSF"].fillna(0) > 0).astype(int)
    df["HasOpenPorch"] = (df["OpenPorchSF"].fillna(0) > 0).astype(int)
    df["OverallQualGrLivArea"] = df["OverallQual"].fillna(0) * df["GrLivArea"].fillna(0)
    df["OverallQualFinishedSF"] = df["OverallQual"].fillna(0) * df["TotalFinishedSF"]
    df["ExterQualTotalSF"] = df["ExterQualScore"] * df["TotalSF"]
    df["KitchenQualTotalBath"] = df["KitchenQualScore"] * df["TotalBath"]
    df["BsmtQualTotalBsmtSF"] = (
        df["BsmtQualScore"] * df["TotalBsmtSF"].fillna(0)
    )
    df["GarageQualGarageArea"] = df["GarageQualScore"] * df["GarageArea"].fillna(0)
    df["OverallCondHouseAge"] = df["OverallCond"].fillna(0) * df["HouseAge"]
    df["OverallQualRemodAge"] = df["OverallQual"].fillna(0) * df["RemodAge"]
    df["OldRemodeled"] = (
        df["HouseAge"].gt(40) & df["Remodeled"].eq(1)
    ).astype(int)
    df["NewHighQual"] = (
        df["NewerDwelling"].eq(1) & df["OverallQual"].ge(7)
    ).astype(int)
    df["FinishedBsmtRatio"] = safe_divide(
        df["BsmtFinSF1"].fillna(0) + df["BsmtFinSF2"].fillna(0),
        df["TotalBsmtSF"].fillna(0),
    ).fillna(0)
    df["GarageAreaPerCar"] = safe_divide(
        df["GarageArea"].fillna(0), df["GarageCars"].fillna(0)
    ).fillna(0)
    df["BathPerRoom"] = safe_divide(
        df["TotalBath"], df["TotRmsAbvGrd"].fillna(0)
    ).fillna(0)
    df["BedroomRatio"] = safe_divide(
        df["BedroomAbvGr"].fillna(0), df["TotRmsAbvGrd"].fillna(0)
    ).fillna(0)
    df["PorchRatio"] = safe_divide(df["TotalPorchSF"], df["TotalSF"]).fillna(0)
    df["GarageAreaRatio"] = safe_divide(
        df["GarageArea"].fillna(0), df["GrLivArea"].fillna(0)
    ).fillna(0)
    df["BsmtRatio"] = safe_divide(
        df["TotalBsmtSF"].fillna(0), df["TotalSF"]
    ).fillna(0)

    log_cols = [
        "LotArea",
        "GrLivArea",
        "TotalSF",
        "TotalFinishedSF",
        "TotalBsmtSF",
        "1stFlrSF",
        "2ndFlrSF",
        "GarageArea",
        "MasVnrArea",
        "WoodDeckSF",
        "OpenPorchSF",
        "TotalPorchSF",
        "TotalOutdoorSF",
        "QualSF",
        "OverallQualGrLivArea",
        "OverallQualFinishedSF",
    ]
    for col in log_cols:
        if col in df.columns:
            df[f"Log{col}"] = np.log1p(df[col].fillna(0).clip(lower=0))

    return df


# Nama alias dipertahankan agar bagian model lama tetap kompatibel.
add_engineered_features = engineer_features

X_train_fe = engineer_features(X_train)
X_valid_fe = engineer_features(X_valid)
X_test_final_fe = engineer_features(X_test_final)

engineered_features = [
    "TotalSF", "TotalBath", "TotalFinishedSF", "TotalOutdoorSF",
    "HouseAge", "RemodAge", "GarageAge", "EffectiveAge",
    "HasPool", "HasGarage", "HasBasement", "HasFireplace",
    "QualSF", "QualBath", "OverallQualGrLivArea",
    "FinishedBsmtRatio", "GarageAreaPerCar", "LogGrLivArea",
]
X_train_fe[engineered_features].head()


### Penentuan Kolom Encoding

Encoding dibedakan menjadi dua:

- **Ordinal encoding** untuk fitur yang memiliki urutan kualitas atau kondisi.
- **One-hot encoding** untuk fitur nominal yang tidak memiliki urutan alami.
- `MSSubClass` diperlakukan sebagai kategori karena nilainya merupakan kode kelas rumah.


In [ ]:
ordinal_features = [col for col in ORDINAL_MAPS if col in X_train_fe.columns]
ordinal_categories = [ORDINAL_MAPS[col] for col in ordinal_features]

categorical_features = X_train_fe.select_dtypes(include=["object"]).columns.tolist()
onehot_features = [col for col in categorical_features if col not in ordinal_features]
numeric_features = [
    col for col in X_train_fe.columns
    if col not in ordinal_features + onehot_features
]

print("Jumlah fitur numerik :", len(numeric_features))
print("Jumlah fitur ordinal :", len(ordinal_features))
print("Jumlah fitur nominal :", len(onehot_features))


### Pipeline Preprocessing Bersama

Pipeline ini digunakan oleh Decision Tree, KNN, dan XGBoost:

1. Missing value numerik diisi dengan median.
2. Fitur numerik dinormalisasi menggunakan `StandardScaler` agar kompatibel dengan model sensitif skala seperti KNN.
3. Missing value ordinal diisi dengan `None`, lalu dilakukan ordinal encoding.
4. Missing value nominal diisi dengan kategori `Missing`, lalu dilakukan one-hot encoding.

Pipeline hanya di-`fit` pada data training untuk menghindari data leakage. Scaling tidak mengubah split pada model tree, sehingga matriks yang sama tetap dapat digunakan oleh Decision Tree dan XGBoost.


In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

ordinal_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="None")),
    ("encoder", OrdinalEncoder(
        categories=ordinal_categories,
        handle_unknown="use_encoded_value",
        unknown_value=-1,
    )),
])

try:
    onehot_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    onehot_encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

nominal_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("encoder", onehot_encoder),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("ord", ordinal_transformer, ordinal_features),
        ("nom", nominal_transformer, onehot_features),
    ],
    remainder="drop",
)

X_train_prepared = preprocessor.fit_transform(X_train_fe)
X_valid_prepared = preprocessor.transform(X_valid_fe)
X_test_final_prepared = preprocessor.transform(X_test_final_fe)

print("X_train_prepared:", X_train_prepared.shape)
print("X_valid_prepared:", X_valid_prepared.shape)
print("X_test_final_prepared:", X_test_final_prepared.shape)


### Hasil Preprocessing dalam Bentuk DataFrame

Cell berikut mengubah hasil preprocessing menjadi `DataFrame` agar nama fitur hasil normalisasi dan encoding bisa diperiksa. Data ini siap dipakai pada tahap training model.

In [ ]:
feature_names = preprocessor.get_feature_names_out()

X_train_prepared_df = pd.DataFrame(X_train_prepared, columns=feature_names, index=X_train.index)
X_valid_prepared_df = pd.DataFrame(X_valid_prepared, columns=feature_names, index=X_valid.index)
X_test_final_prepared_df = pd.DataFrame(X_test_final_prepared, columns=feature_names, index=X_test_final.index)

X_train_prepared_df.head()

In [ ]:
preprocessing_report = pd.DataFrame({
    'tahap': [
        'Data awal training',
        'Setelah feature engineering',
        'Setelah normalisasi dan encoding'
    ],
    'jumlah_baris': [X_train.shape[0], X_train_fe.shape[0], X_train_prepared_df.shape[0]],
    'jumlah_fitur': [X_train.shape[1], X_train_fe.shape[1], X_train_prepared_df.shape[1]]
})

preprocessing_report

Sampai tahap ini, seluruh model menggunakan hasil preprocessing yang sama:

- `X_train_prepared` atau `X_train_prepared_df`
- `X_valid_prepared` atau `X_valid_prepared_df`
- `X_test_final_prepared` atau `X_test_final_prepared_df`
- `y_train` dan `y_valid`

Perbedaan setelah tahap ini hanya terdapat pada model, transformasi target, feature selection, dan hyperparameter masing-masing model.


## 2. Training Model - Decision Tree Regressor (Willy Marcelius - 5025241096)

Bagian ini mengerjakan eksperimen model **Decision Tree Regressor** untuk memprediksi `SalePrice`. Eksperimen dibuat untuk membandingkan performa model pada kondisi normal dan pada dua skenario uji coba yang diusung kelompok:

1. **Feature Selection**: model hanya menggunakan sejumlah fitur terbaik berdasarkan skor `mutual_info_regression`.
2. **Label Price Logaritmik**: target `SalePrice` ditransformasi dengan `log1p` saat training, lalu prediksi dikembalikan ke skala harga asli dengan `expm1`.

Agar perbandingan lebih lengkap, eksperimen juga menyertakan kombinasi kedua skenario tersebut.

### Setup Eksperimen

Metrik evaluasi yang digunakan adalah:

- **RMSE**: menunjukkan rata-rata besar error dalam satuan harga rumah, dengan penalti lebih besar untuk error besar.
- **MAE**: menunjukkan rata-rata absolut error dalam satuan harga rumah.
- **RMSLE**: cocok untuk target harga karena mengukur error relatif pada skala logaritmik.
- **R2 Score**: menunjukkan proporsi variasi target yang mampu dijelaskan model.

Semua eksperimen menggunakan data training dan validation split yang sama dari tahap preprocessing.

In [ ]:
from sklearn.feature_selection import SelectKBest, mutual_info_regression
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_squared_log_error, r2_score
from sklearn.tree import DecisionTreeRegressor

try:
    import matplotlib.pyplot as plt
    HAS_MATPLOTLIB = True
except ModuleNotFoundError:
    HAS_MATPLOTLIB = False
    print('matplotlib belum tersedia, sehingga cell grafik akan dilewati. Tabel performa tetap dapat digunakan.')

RANDOM_STATE = 42


def evaluate_regression(y_true, y_pred):
    """Menghitung metrik regresi pada skala harga asli."""
    y_pred_safe = np.clip(y_pred, 0, None)
    return {
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSLE': np.sqrt(mean_squared_log_error(y_true, y_pred_safe)),
        'R2': r2_score(y_true, y_pred)
    }


def get_candidate_k_values(n_features):
    """Menentukan pilihan jumlah fitur untuk skenario feature selection."""
    base_values = [20, 40, 60, 80, 120, 160]
    k_values = sorted({k for k in base_values if k < n_features})
    k_values.append('all')
    return k_values


param_grid = [
    {
        'max_depth': max_depth,
        'min_samples_split': min_samples_split,
        'min_samples_leaf': min_samples_leaf,
        'max_features': max_features
    }
    for max_depth in [4, 6, 8, 10, None]
    for min_samples_split in [2, 10, 20]
    for min_samples_leaf in [1, 3, 5, 10]
    for max_features in [None, 'sqrt']
]

print('Jumlah kombinasi hyperparameter:', len(param_grid))
print('Jumlah fitur setelah preprocessing:', X_train_prepared_df.shape[1])

### Fungsi Training dan Evaluasi

Fungsi berikut menjalankan satu eksperimen Decision Tree Regressor. Jika `use_feature_selection=True`, jumlah fitur terbaik ikut dicari dari beberapa kandidat `k`. Jika `use_log_target=True`, model dilatih pada target `log1p(SalePrice)` dan hasil prediksi dikembalikan ke skala harga asli sebelum evaluasi.

In [ ]:
def run_decision_tree_experiment(name, use_feature_selection=False, use_log_target=False):
    y_train_model = np.log1p(y_train) if use_log_target else y_train
    selection_target = y_train_model if use_log_target else y_train

    k_candidates = get_candidate_k_values(X_train_prepared_df.shape[1]) if use_feature_selection else ['all']
    best_result = None

    for k in k_candidates:
        if use_feature_selection and k != 'all':
            selector = SelectKBest(
                score_func=lambda X_data, y_data: mutual_info_regression(
                    X_data,
                    y_data,
                    random_state=RANDOM_STATE
                ),
                k=k
            )
            X_train_model = selector.fit_transform(X_train_prepared_df, selection_target)
            X_valid_model = selector.transform(X_valid_prepared_df)
            selected_features = X_train_prepared_df.columns[selector.get_support()].tolist()
        else:
            selector = None
            X_train_model = X_train_prepared_df
            X_valid_model = X_valid_prepared_df
            selected_features = X_train_prepared_df.columns.tolist()

        for params in param_grid:
            model = DecisionTreeRegressor(random_state=RANDOM_STATE, **params)
            model.fit(X_train_model, y_train_model)
            y_pred_model = model.predict(X_valid_model)
            y_pred = np.expm1(y_pred_model) if use_log_target else y_pred_model

            metrics = evaluate_regression(y_valid, y_pred)
            result = {
                'Skenario': name,
                'Feature Selection': 'Ya' if use_feature_selection else 'Tidak',
                'Label Logaritmik': 'Ya' if use_log_target else 'Tidak',
                'Jumlah Fitur': len(selected_features),
                'RMSE': metrics['RMSE'],
                'MAE': metrics['MAE'],
                'RMSLE': metrics['RMSLE'],
                'R2': metrics['R2'],
                'Best Params': params,
                'Selected Features': selected_features,
                'Model': model,
                'Selector': selector,
                'Prediction': y_pred
            }

            if best_result is None or result['RMSE'] < best_result['RMSE']:
                best_result = result

    return best_result

### Tahap Training Model

Pada tahap ini, Decision Tree Regressor dilatih menggunakan beberapa konfigurasi skenario. Setiap skenario menggunakan data training yang sama, kemudian hasilnya dibandingkan pada validation set.

### Menjalankan Seluruh Skenario Uji Coba

Empat eksperimen berikut digunakan agar dampak setiap skenario dapat dibandingkan dengan jelas:

1. **Baseline**: tanpa feature selection dan tanpa transformasi log target.
2. **Feature Selection**: hanya menggunakan fitur terpilih.
3. **Label Logaritmik**: target harga ditransformasi logaritmik.
4. **Feature Selection + Label Logaritmik**: kombinasi kedua skenario.

In [ ]:
experiments = [
    ('Baseline Decision Tree', False, False),
    ('Feature Selection', True, False),
    ('Label Price Logaritmik', False, True),
    ('Feature Selection + Label Logaritmik', True, True)
]

experiment_results = []

for experiment_name, use_feature_selection, use_log_target in experiments:
    print(f'Menjalankan eksperimen: {experiment_name}')
    result = run_decision_tree_experiment(
        experiment_name,
        use_feature_selection=use_feature_selection,
        use_log_target=use_log_target
    )
    experiment_results.append(result)

print('Seluruh eksperimen selesai.')

### Tahap Testing dan Evaluasi pada Validation Set

Tahap testing dilakukan dengan memprediksi data validation (`X_valid_prepared_df`) yang tidak digunakan saat model dilatih. Hasil prediksi dibandingkan dengan `y_valid`, lalu dievaluasi menggunakan RMSE, MAE, RMSLE, dan R2.

### Tabel Hasil Performa Tiap Skenario

Tabel berikut menunjukkan performa terbaik dari masing-masing skenario berdasarkan pencarian hyperparameter sederhana. Model terbaik dipilih berdasarkan nilai **RMSE** terendah pada validation set.

In [ ]:
results_df = pd.DataFrame(experiment_results)

performance_columns = [
    'Skenario', 'Feature Selection', 'Label Logaritmik', 'Jumlah Fitur',
    'RMSE', 'MAE', 'RMSLE', 'R2', 'Best Params'
]

performance_table = results_df[performance_columns].copy()
performance_table[['RMSE', 'MAE', 'RMSLE', 'R2']] = performance_table[['RMSE', 'MAE', 'RMSLE', 'R2']].round(4)
performance_table = performance_table.sort_values('RMSE').reset_index(drop=True)
performance_table

### Visualisasi Perbandingan Performa

Grafik berikut membandingkan performa setiap skenario berdasarkan RMSE, MAE, RMSLE, dan R2. Untuk RMSE, MAE, dan RMSLE, nilai yang lebih kecil lebih baik. Untuk R2, nilai yang lebih besar lebih baik.

In [ ]:
plot_df = performance_table.set_index('Skenario')
metrics_to_plot = ['RMSE', 'MAE', 'RMSLE', 'R2']

if HAS_MATPLOTLIB:
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    axes = axes.ravel()

    for ax, metric in zip(axes, metrics_to_plot):
        plot_df[metric].plot(kind='bar', ax=ax, color='#4C78A8')
        ax.set_title(metric)
        ax.set_xlabel('')
        ax.tick_params(axis='x', rotation=25)
        ax.grid(axis='y', linestyle='--', alpha=0.35)

    plt.tight_layout()
    plt.show()
else:
    print('Grafik tidak ditampilkan karena matplotlib belum tersedia.')
    display(performance_table)

### Feature Selection: Fitur yang Terpilih

Cell berikut menampilkan fitur yang digunakan oleh skenario dengan feature selection. Daftar ini membantu mendokumentasikan fitur mana yang dianggap paling informatif untuk Decision Tree Regressor berdasarkan `mutual_info_regression`.

In [ ]:
feature_selection_rows = []

for result in experiment_results:
    if result['Feature Selection'] == 'Ya':
        for rank, feature_name in enumerate(result['Selected Features'], start=1):
            feature_selection_rows.append({
                'Skenario': result['Skenario'],
                'Rank': rank,
                'Fitur Terpilih': feature_name
            })

selected_features_df = pd.DataFrame(feature_selection_rows)
selected_features_df.head(30)

### Prediksi vs Nilai Aktual untuk Model Terbaik

Visualisasi berikut digunakan untuk melihat pola kesalahan model terbaik. Titik yang semakin dekat dengan garis diagonal menunjukkan prediksi yang semakin mendekati harga aktual.

In [ ]:
best_result = min(experiment_results, key=lambda item: item['RMSE'])

if HAS_MATPLOTLIB:
    plt.figure(figsize=(7, 6))
    plt.scatter(y_valid, best_result['Prediction'], alpha=0.65, color='#2F855A')
    min_price = min(y_valid.min(), best_result['Prediction'].min())
    max_price = max(y_valid.max(), best_result['Prediction'].max())
    plt.plot([min_price, max_price], [min_price, max_price], color='#C53030', linestyle='--')
    plt.xlabel('SalePrice Aktual')
    plt.ylabel('SalePrice Prediksi')
    plt.title(f"Prediksi vs Aktual - {best_result['Skenario']}")
    plt.grid(linestyle='--', alpha=0.35)
    plt.show()
else:
    print('Scatter plot prediksi vs aktual tidak ditampilkan karena matplotlib belum tersedia.')

print('Model terbaik:', best_result['Skenario'])
print('RMSE terbaik :', round(best_result['RMSE'], 4))
print('Parameter    :', best_result['Best Params'])

### Pembahasan Hasil Uji Coba

Berdasarkan tabel dan grafik performa di atas, beberapa hal yang perlu diperhatikan adalah:

- **Baseline Decision Tree** menjadi pembanding utama karena model dilatih tanpa feature selection dan tanpa transformasi target. Performa baseline menunjukkan kemampuan awal Decision Tree setelah preprocessing.
- **Skenario Feature Selection** menguji apakah pengurangan fitur membantu model menghindari noise dari fitur hasil encoding yang jumlahnya cukup banyak. Jika RMSE dan RMSLE lebih rendah dari baseline, berarti fitur terpilih membantu model menjadi lebih stabil. Jika performanya turun, berarti sebagian informasi penting kemungkinan ikut terbuang saat seleksi fitur.
- **Skenario Label Price Logaritmik** menguji apakah transformasi target membantu model menangani distribusi harga rumah yang cenderung skewed. Jika RMSLE membaik, transformasi log membuat model lebih baik dalam menangkap perbedaan relatif antar harga rumah.
- **Skenario Kombinasi** menunjukkan apakah feature selection dan transformasi log saling melengkapi. Kombinasi ini tidak selalu otomatis menjadi yang terbaik karena Decision Tree sensitif terhadap pemilihan fitur dan kedalaman pohon.
- Model dengan **RMSE terendah** dipilih sebagai model Decision Tree Regressor terbaik pada validation set. Namun, metrik lain tetap perlu dilihat: MAE membantu membaca error rata-rata yang lebih mudah dipahami, RMSLE penting untuk konteks harga, dan R2 menunjukkan daya jelaskan model.

Secara umum, Decision Tree Regressor mudah diinterpretasikan, tetapi rentan overfitting. Karena itu, pembatasan seperti `max_depth`, `min_samples_split`, dan `min_samples_leaf` digunakan dalam eksperimen agar pohon tidak terlalu mengikuti data training secara berlebihan.

### Kesimpulan Bagian Decision Tree Regressor

Kesimpulan akhir untuk bagian ini dapat diisi berdasarkan baris terbaik pada `performance_table`. Skenario dengan RMSE terendah menjadi kandidat utama model Decision Tree Regressor terbaik, sedangkan perbandingan terhadap baseline menunjukkan apakah feature selection, label logaritmik, atau kombinasi keduanya benar-benar memberi peningkatan performa.

### Tahap Testing pada Data Test

Setelah skenario terbaik ditentukan dari validation set, konfigurasi terbaik tersebut digunakan kembali untuk melatih model pada seluruh data training. Langkah ini umum dilakukan sebelum membuat submission karena seluruh data berlabel dimanfaatkan untuk training akhir.

Pada tahap ini, model memprediksi `SalePrice` untuk `test.csv`. Karena data test Kaggle tidak memiliki label asli, evaluasi akhirnya diperoleh dari public score Kaggle setelah file submission diupload.

File submission dibuat mengikuti format `sample_submission.csv`, yaitu kolom `Id` dan `SalePrice`.

In [ ]:
from pathlib import Path

best_scenario = best_result['Skenario']
best_params = best_result['Best Params']
use_best_feature_selection = best_result['Feature Selection'] == 'Ya'
use_best_log_target = best_result['Label Logaritmik'] == 'Ya'

print('Skenario terbaik untuk submission:', best_scenario)
print('Menggunakan feature selection:', use_best_feature_selection)
print('Menggunakan label logaritmik:', use_best_log_target)
print('Parameter terbaik:', best_params)

In [ ]:
# Refit preprocessing pada seluruh data train agar model final memakai semua data berlabel.
final_preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('ord', ordinal_transformer, ordinal_features),
        ('nom', nominal_transformer, onehot_features)
    ],
    remainder='drop'
)

X_full_fe = add_engineered_features(X)
X_test_submission_fe = add_engineered_features(X_test_final)

X_full_prepared = final_preprocessor.fit_transform(X_full_fe)
X_test_submission_prepared = final_preprocessor.transform(X_test_submission_fe)

final_feature_names = final_preprocessor.get_feature_names_out()
X_full_prepared_df = pd.DataFrame(X_full_prepared, columns=final_feature_names, index=X.index)
X_test_submission_prepared_df = pd.DataFrame(
    X_test_submission_prepared,
    columns=final_feature_names,
    index=X_test_final.index
)

print('X_full_prepared:', X_full_prepared_df.shape)
print('X_test_submission_prepared:', X_test_submission_prepared_df.shape)

In [ ]:
y_full_model = np.log1p(y) if use_best_log_target else y

if use_best_feature_selection:
    final_k = best_result['Selector'].k
    final_selector = SelectKBest(
        score_func=lambda X_data, y_data: mutual_info_regression(
            X_data,
            y_data,
            random_state=RANDOM_STATE
        ),
        k=final_k
    )
    final_X_train = final_selector.fit_transform(X_full_prepared_df, y_full_model)
    final_X_test = final_selector.transform(X_test_submission_prepared_df)
    final_selected_features = X_full_prepared_df.columns[final_selector.get_support()].tolist()
else:
    final_selector = None
    final_X_train = X_full_prepared_df
    final_X_test = X_test_submission_prepared_df
    final_selected_features = X_full_prepared_df.columns.tolist()

final_model = DecisionTreeRegressor(random_state=RANDOM_STATE, **best_params)
final_model.fit(final_X_train, y_full_model)

final_test_prediction_model = final_model.predict(final_X_test)
final_test_prediction = np.expm1(final_test_prediction_model) if use_best_log_target else final_test_prediction_model
final_test_prediction = np.clip(final_test_prediction, 0, None)

print('Jumlah fitur model final:', len(final_selected_features))
print('Contoh prediksi:', final_test_prediction[:5])

In [ ]:
submission = pd.DataFrame({'Id': test_df['Id']})
submission['SalePrice'] = final_test_prediction

submission_path = Path('submission_decision_tree_willy.csv')
submission.to_csv(submission_path, index=False)

submission.head()

## 3. Training Model - K-Nearest Neighbors Regressor (Mario Napitupulu - 5025241085)

Bagian ini mengerjakan eksperimen model **K-Nearest Neighbors (KNN) Regressor** untuk memprediksi `SalePrice`. Karena KNN menghitung jarak (*Euclidean/Manhattan distance*) antar data, dataset yang sudah dinormalisasi pada tahap preprocessing akan membuat algoritma ini bekerja optimal.

Sama seperti eksperimen sebelumnya, KNN akan diuji menggunakan empat skenario yang diusung kelompok:
1. **Baseline**: Menggunakan seluruh fitur yang sudah dinormalisasi.
2. **Feature Selection**: Memilih sejumlah fitur terbaik berdasarkan skor `mutual_info_regression`.
3. **Label Price Logaritmik**: Target `SalePrice` ditransformasi dengan `log1p` saat training.
4. **Feature Selection + Label Logaritmik**: Kombinasi kedua skenario.

In [3]:
from sklearn.neighbors import KNeighborsRegressor

# 1. Setup hyperparameter grid untuk KNN
knn_param_grid = [
    {
        'n_neighbors': n_neighbors,
        'weights': weights,
        'p': p
    }
    for n_neighbors in [3, 5, 7, 9, 11, 15]
    for weights in ['uniform', 'distance']
    for p in [1, 2] # p=1: Manhattan, p=2: Euclidean
]

print('Jumlah kombinasi hyperparameter KNN:', len(knn_param_grid))

# 2. Fungsi utama untuk menjalankan eksperimen KNN (Meminjam struktur milik Willy)
def run_knn_experiment(name, use_feature_selection=False, use_log_target=False):
    y_train_model = np.log1p(y_train) if use_log_target else y_train
    selection_target = y_train_model if use_log_target else y_train

    k_candidates = get_candidate_k_values(X_train_prepared_df.shape[1]) if use_feature_selection else ['all']
    best_result = None

    for k in k_candidates:
        if use_feature_selection and k != 'all':
            selector = SelectKBest(
                score_func=lambda X_data, y_data: mutual_info_regression(X_data, y_data, random_state=RANDOM_STATE),
                k=k
            )
            X_train_model = selector.fit_transform(X_train_prepared_df, selection_target)
            X_valid_model = selector.transform(X_valid_prepared_df)
            selected_features = X_train_prepared_df.columns[selector.get_support()].tolist()
        else:
            selector = None
            X_train_model = X_train_prepared_df
            X_valid_model = X_valid_prepared_df
            selected_features = X_train_prepared_df.columns.tolist()

        for params in knn_param_grid:
            model = KNeighborsRegressor(**params)
            model.fit(X_train_model, y_train_model)
            y_pred_model = model.predict(X_valid_model)
            y_pred = np.expm1(y_pred_model) if use_log_target else y_pred_model

            metrics = evaluate_regression(y_valid, y_pred)
            result = {
                'Skenario': name,
                'Feature Selection': 'Ya' if use_feature_selection else 'Tidak',
                'Label Logaritmik': 'Ya' if use_log_target else 'Tidak',
                'Jumlah Fitur': len(selected_features),
                'RMSE': metrics['RMSE'],
                'MAE': metrics['MAE'],
                'RMSLE': metrics['RMSLE'],
                'R2': metrics['R2'],
                'Best Params': params,
                'Selected Features': selected_features,
                'Model': model,
                'Selector': selector,
                'Prediction': y_pred
            }

            if best_result is None or result['RMSE'] < best_result['RMSE']:
                best_result = result

    return best_result

Jumlah kombinasi hyperparameter KNN: 24


### Menjalankan Seluruh Skenario Uji Coba (KNN)
Empat eksperimen berikut digunakan agar dampak setiap skenario dapat dibandingkan dengan jelas:
* **Baseline**: tanpa feature selection dan tanpa transformasi log target.
* **Feature Selection**: hanya menggunakan fitur terpilih.
* **Label Logaritmik**: target harga ditransformasi logaritmik.
* **Feature Selection + Label Logaritmik**: kombinasi kedua skenario.

In [4]:
knn_experiments = [
    ('Baseline KNN', False, False),
    ('Feature Selection', True, False),
    ('Label Price Logaritmik', False, True),
    ('Feature Selection + Label Logaritmik', True, True)
]

knn_experiment_results = []

for experiment_name, use_feature_selection, use_log_target in knn_experiments:
    print(f'Menjalankan eksperimen: {experiment_name}')
    result = run_knn_experiment(
        experiment_name,
        use_feature_selection=use_feature_selection,
        use_log_target=use_log_target
    )
    knn_experiment_results.append(result)

print('Seluruh eksperimen KNN selesai.')

Menjalankan eksperimen: Baseline KNN


NameError: name 'y_train' is not defined

### Tahap Testing dan Evaluasi pada Validation Set
Tahap testing dilakukan dengan memprediksi data validation (`X_valid_prepared_df`) yang tidak digunakan saat model dilatih. Hasil prediksi dibandingkan dengan `y_valid`, lalu dievaluasi menggunakan RMSE, MAE, RMSLE, dan R2.

#### Tabel Hasil Performa Tiap Skenario (KNN)
Tabel berikut menunjukkan performa terbaik model KNN dari masing-masing skenario berdasarkan pencarian hyperparameter. Model terbaik dipilih berdasarkan nilai RMSE terendah pada validation set.

In [ ]:
import pandas as pd
from IPython.display import display

knn_results_df = pd.DataFrame(knn_experiment_results)

performance_columns = [
    'Skenario', 'Feature Selection', 'Label Logaritmik', 'Jumlah Fitur',
    'RMSE', 'MAE', 'RMSLE', 'R2', 'Best Params'
]

knn_performance_table = knn_results_df[performance_columns].copy()
knn_performance_table[['RMSE', 'MAE', 'RMSLE', 'R2']] = knn_performance_table[['RMSE', 'MAE', 'RMSLE', 'R2']].round(4)
knn_performance_table = knn_performance_table.sort_values('RMSE').reset_index(drop=True)

display(knn_performance_table)

#### Visualisasi Perbandingan Performa (KNN)
Grafik berikut membandingkan performa setiap skenario berdasarkan RMSE, MAE, RMSLE, dan R2. Untuk RMSE, MAE, dan RMSLE, nilai yang lebih kecil lebih baik. Untuk R2, nilai yang lebih besar lebih baik.

In [5]:
import matplotlib.pyplot as plt
from IPython.display import display

plot_df = knn_performance_table.set_index('Skenario')
metrics_to_plot = ['RMSE', 'MAE', 'RMSLE', 'R2']

# Membuat grid 2x2 untuk grafik
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.ravel()

for ax, metric in zip(axes, metrics_to_plot):
    plot_df[metric].plot(kind='bar', ax=ax, color='#4C78A8')
    ax.set_title(metric)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=25)
    ax.grid(axis='y', linestyle='--', alpha=0.35)

plt.tight_layout()
plt.show()

NameError: name 'knn_performance_table' is not defined

#### Feature Selection: Fitur yang Terpilih (KNN)
Cell berikut menampilkan fitur yang digunakan oleh skenario dengan feature selection. Daftar ini membantu mendokumentasikan fitur mana yang dianggap paling informatif untuk K-Nearest Neighbors Regressor berdasarkan `mutual_info_regression`.

In [ ]:
import pandas as pd
from IPython.display import display

knn_feature_selection_rows = []

for result in knn_experiment_results:
    if result['Feature Selection'] == 'Ya':
        for rank, feature_name in enumerate(result['Selected Features'], start=1):
            knn_feature_selection_rows.append({
                'Skenario': result['Skenario'],
                'Rank': rank,
                'Fitur Terpilih': feature_name
            })

knn_selected_features_df = pd.DataFrame(knn_feature_selection_rows)

# Menampilkan 30 fitur teratas
display(knn_selected_features_df.head(30))

#### Prediksi vs Nilai Aktual untuk Model Terbaik (KNN)
Visualisasi berikut digunakan untuk melihat pola kesalahan model KNN terbaik. Titik yang semakin dekat dengan garis diagonal menunjukkan prediksi yang semakin mendekati harga aktual.

In [6]:
import matplotlib.pyplot as plt

# Mencari model dengan nilai RMSE paling kecil (paling bagus)
best_knn_result = min(knn_experiment_results, key=lambda item: item['RMSE'])

plt.figure(figsize=(7, 6))
plt.scatter(y_valid, best_knn_result['Prediction'], alpha=0.65, color='#2F855A')

# Membuat garis putus-putus merah sebagai batas ideal (prediksi = aktual)
min_price = min(y_valid.min(), best_knn_result['Prediction'].min())
max_price = max(y_valid.max(), best_knn_result['Prediction'].max())
plt.plot([min_price, max_price], [min_price, max_price], color='#C53030', linestyle='--')

plt.xlabel('SalePrice Aktual')
plt.ylabel('SalePrice Prediksi')
plt.title(f"Prediksi vs Aktual - {best_knn_result['Skenario']}")
plt.grid(linestyle='--', alpha=0.35)
plt.show()

print('Model terbaik KNN:', best_knn_result['Skenario'])
print('RMSE terbaik     :', round(best_knn_result['RMSE'], 4))
print('Parameter        :', best_knn_result['Best Params'])

ValueError: min() iterable argument is empty

### Pembahasan Hasil Uji Coba (KNN)
Berdasarkan tabel dan grafik performa di atas, beberapa hal yang perlu diperhatikan adalah:
* **Baseline KNN** menjadi pembanding utama karena model dilatih dengan seluruh fitur (yang sudah dinormalisasi) namun tanpa seleksi fitur maupun transformasi target. Performa baseline menunjukkan kemampuan dasar KNN dalam menghitung jarak antar rumah di ruang berdimensi tinggi.
* **Skenario Feature Selection** menguji apakah pengurangan fitur membantu model menghindari *Curse of Dimensionality* (kutukan dimensi tinggi). Karena KNN sangat bergantung pada perhitungan jarak (seperti Euclidean), membuang fitur yang tidak relevan biasanya akan membuat perhitungan jarak menjadi lebih akurat dan RMSE menurun.
* **Skenario Label Price Logaritmik** menguji apakah transformasi target membantu KNN menangani target harga rumah yang rentangnya sangat jauh/jomplang. Transformasi log seringkali membantu model lebih baik dalam menangkap pola harga yang tidak terdistribusi normal.
* **Skenario Kombinasi** menunjukkan apakah seleksi fitur dan transformasi logaritmik saling melengkapi untuk menghasilkan prediksi jarak terdekat yang paling optimal.

### Kesimpulan Bagian K-Nearest Neighbors Regressor
Kesimpulan akhir untuk bagian KNN ini didasarkan pada baris terbaik pada tabel performa. Skenario dengan metrik RMSE terendah dan nilai $R^2$ tertinggi menjadi kandidat utama model K-Nearest Neighbors Regressor terbaik. Perbandingan performa ini nantinya akan disandingkan dengan model dari anggota kelompok lain (seperti Decision Tree) untuk melihat apakah pendekatan berbasis perhitungan jarak (KNN) lebih unggul dibandingkan pendekatan berbasis aturan pohon pada dataset harga rumah ini.

### Tahap Testing pada Data Test (KNN Regressor)
Setelah skenario terbaik ditentukan dari validation set, konfigurasi terbaik tersebut digunakan untuk memprediksi data test (`test.csv`). Karena data test Kaggle tidak memiliki label asli, evaluasi akhirnya diperoleh dari public score Kaggle setelah file submission diupload.

File submission dibuat mengikuti format `sample_submission.csv`, yaitu hanya berisi kolom `Id` dan `SalePrice`.

In [ ]:
import pandas as pd
import numpy as np

# 1. Mengambil model dan selector (jika ada) dari skenario KNN terbaik
best_knn_model = best_knn_result['Model']
best_knn_selector = best_knn_result['Selector']
knn_used_log_target = best_knn_result['Label Logaritmik'] == 'Ya'

# 2. Menyiapkan data test (menerapkan Feature Selection jika skenario terbaik menggunakannya)
if best_knn_selector is not None:
    X_test_knn = best_knn_selector.transform(X_test_prepared_df)
else:
    X_test_knn = X_test_prepared_df

# 3. Melakukan prediksi pada data test
y_pred_test_knn_raw = best_knn_model.predict(X_test_knn)

# 4. Mengembalikan skala harga ke aslinya (jika sebelumnya menggunakan log1p)
if knn_used_log_target:
    y_pred_test_knn = np.expm1(y_pred_test_knn_raw)
else:
    y_pred_test_knn = y_pred_test_knn_raw

# 5. Memasukkan hasil prediksi ke format Kaggle
sample_submission = pd.read_csv('sample_submission.csv')
submission_knn = sample_submission.copy()
submission_knn['SalePrice'] = y_pred_test_knn

# 6. Menyimpan file CSV
nama_file_submission = 'submission_knn.csv'
submission_knn.to_csv(nama_file_submission, index=False)

print(f"File {nama_file_submission} berhasil dibuat dan siap di-submit ke Kaggle!")
display(submission_knn.head())

## 4. Training Model - XGBoost Regressor (Hajendra Herlambang - 5025241101)

Bagian ini mengerjakan eksperimen model **XGBoost Regressor** untuk memprediksi `SalePrice`. Eksperimen dibuat untuk membandingkan performa baseline, penggunaan label logaritmik, dan konfigurasi hyperparameter yang sudah dituning. Semua skenario menggunakan preprocessing yang sama agar perbandingan tetap adil.


### Setup Eksperimen

Metrik evaluasi yang digunakan adalah:

- **RMSE**: menunjukkan rata-rata besar error dalam satuan harga rumah, dengan penalti lebih besar untuk error besar.
- **MAE**: menunjukkan rata-rata absolut selisih harga prediksi dan harga aktual.
- **RMSLE**: mengukur error pada skala logaritmik dan sesuai untuk distribusi harga yang right-skewed.
- **R2**: menunjukkan proporsi variasi harga rumah yang dapat dijelaskan oleh model.

Nilai RMSE, MAE, dan RMSLE yang lebih kecil menunjukkan performa yang lebih baik, sedangkan nilai R2 yang lebih besar menunjukkan performa yang lebih baik.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_squared_log_error, r2_score
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print("XGBoost menggunakan preprocessing bersama.")
print("XGBoost train shape:", X_train_prepared_df.shape)
print("XGBoost validation shape:", X_valid_prepared_df.shape)


### Fungsi Training dan Evaluasi

Fungsi berikut menjalankan satu eksperimen XGBoost Regressor menggunakan hasil preprocessing bersama. Jika `use_log_target=True`, model dilatih menggunakan `log1p(SalePrice)` dan hasil prediksi dikembalikan ke skala harga asli menggunakan `expm1`.


In [ ]:
def evaluate_xgboost(y_true, y_pred):
    y_pred = np.clip(y_pred, 0, None)
    return {
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSLE": np.sqrt(mean_squared_log_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }


def run_xgboost_experiment(name, use_log_target, model_params):
    y_train_model = np.log1p(y_train) if use_log_target else y_train
    model = XGBRegressor(**model_params)
    model.fit(X_train_prepared_df, y_train_model)

    prediction_model = model.predict(X_valid_prepared_df)
    prediction = np.expm1(prediction_model) if use_log_target else prediction_model
    prediction = np.clip(prediction, 0, None)
    metrics = evaluate_xgboost(y_valid, prediction)

    return {
        "Skenario": name,
        "Label Logaritmik": "Ya" if use_log_target else "Tidak",
        "RMSE": metrics["RMSE"],
        "MAE": metrics["MAE"],
        "RMSLE": metrics["RMSLE"],
        "R2": metrics["R2"],
        "Params": model_params,
        "Model": model,
        "Prediction": prediction,
        "Actual": y_valid,
    }


### Tahap Training Model

Pada tahap ini, XGBoost Regressor dilatih menggunakan beberapa konfigurasi skenario. Setiap skenario menggunakan data training yang sama, kemudian hasilnya dibandingkan pada validation set yang sama.

### Menjalankan Seluruh Skenario Uji Coba

Tiga eksperimen berikut digunakan agar dampak transformasi target dan tuning hyperparameter dapat dibandingkan dengan jelas:

1. **Baseline XGBoost**: target harga asli dan parameter dasar.
2. **Label Price Logaritmik**: parameter dasar dengan transformasi `log1p(SalePrice)`.
3. **Tuned + Label Logaritmik**: transformasi target logaritmik dan konfigurasi hyperparameter final yang sudah dituning.


In [ ]:
xgb_base_params = {
    "objective": "reg:squarederror",
    "n_estimators": 1000,
    "learning_rate": 0.03,
    "max_depth": 3,
    "min_child_weight": 1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    "random_state": 42,
    "n_jobs": -1,
    "eval_metric": "rmse",
}

xgb_tuned_params = {
    "objective": "reg:squarederror",
    "n_estimators": 3800,
    "learning_rate": 0.015,
    "max_depth": 3,
    "min_child_weight": 2,
    "subsample": 0.60,
    "colsample_bytree": 0.70,
    "reg_alpha": 0.0,
    "reg_lambda": 1.3,
    "gamma": 0.001,
    "random_state": 42,
    "n_jobs": -1,
    "eval_metric": "rmse",
}

xgb_experiments = [
    ("Baseline XGBoost", False, xgb_base_params),
    ("XGBoost + Label Logaritmik", True, xgb_base_params),
    ("XGBoost Tuned + Label Logaritmik", True, xgb_tuned_params),
]

xgb_experiment_results = []
for experiment_name, use_log_target, params in xgb_experiments:
    print("Menjalankan eksperimen:", experiment_name)
    xgb_experiment_results.append(
        run_xgboost_experiment(experiment_name, use_log_target, params)
    )

print("Seluruh eksperimen XGBoost selesai.")


### Tahap Testing dan Evaluasi pada Validation Set

Tahap testing dilakukan dengan memprediksi data validation yang tidak digunakan saat model dilatih. Hasil prediksi dikembalikan ke skala harga asli sebelum dibandingkan dengan harga aktual.

### Tabel Hasil Performa Tiap Skenario

Tabel berikut menunjukkan performa masing-masing skenario. Model terbaik dipilih berdasarkan nilai **RMSLE** terendah karena target harga memiliki distribusi right-skewed dan evaluasi House Prices menggunakan error berbasis logaritmik.


In [ ]:
xgb_results_df = pd.DataFrame(xgb_experiment_results)
xgb_performance_columns = [
    "Skenario", "Label Logaritmik", "RMSE", "MAE", "RMSLE", "R2"
]
xgb_performance_table = (
    xgb_results_df[xgb_performance_columns]
    .sort_values("RMSLE")
    .reset_index(drop=True)
)
xgb_performance_table[["RMSE", "MAE", "RMSLE", "R2"]] = (
    xgb_performance_table[["RMSE", "MAE", "RMSLE", "R2"]].round(4)
)
xgb_performance_table


### Visualisasi Perbandingan Performa

Grafik berikut membandingkan performa setiap skenario berdasarkan RMSE dan RMSLE. Nilai yang lebih kecil menunjukkan hasil prediksi yang lebih baik.


In [ ]:
if "HAS_MATPLOTLIB" in globals() and HAS_MATPLOTLIB:
    xgb_plot = xgb_performance_table.set_index("Skenario")
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    xgb_plot["RMSE"].plot(kind="bar", ax=axes[0], color="#2563EB")
    axes[0].set_title("RMSE XGBoost")
    axes[0].set_ylabel("Harga")
    xgb_plot["RMSLE"].plot(kind="bar", ax=axes[1], color="#16A34A")
    axes[1].set_title("RMSLE XGBoost")
    axes[1].set_ylabel("Log Error")
    plt.tight_layout()
    plt.show()
else:
    print("Matplotlib tidak tersedia; gunakan tabel performa di atas.")


### Prediksi vs Nilai Aktual untuk Model Terbaik

Visualisasi berikut digunakan untuk melihat pola kesalahan model terbaik. Titik yang semakin dekat dengan garis diagonal menunjukkan prediksi yang semakin mendekati harga aktual.


In [ ]:
xgb_best_result = min(xgb_experiment_results, key=lambda item: item["RMSLE"])
print("Skenario terbaik:", xgb_best_result["Skenario"])
print("RMSE:", round(xgb_best_result["RMSE"], 4))
print("RMSLE:", round(xgb_best_result["RMSLE"], 6))
print("R2:", round(xgb_best_result["R2"], 4))

if "HAS_MATPLOTLIB" in globals() and HAS_MATPLOTLIB:
    plt.figure(figsize=(7, 6))
    plt.scatter(
        xgb_best_result["Actual"],
        xgb_best_result["Prediction"],
        alpha=0.65,
        color="#2563EB",
    )
    limits = [
        min(xgb_best_result["Actual"].min(), xgb_best_result["Prediction"].min()),
        max(xgb_best_result["Actual"].max(), xgb_best_result["Prediction"].max()),
    ]
    plt.plot(limits, limits, "r--")
    plt.xlabel("Harga Aktual")
    plt.ylabel("Harga Prediksi")
    plt.title("Prediksi vs Aktual - XGBoost Terbaik")
    plt.show()


### Pembahasan Hasil Uji Coba

Berdasarkan tabel dan grafik performa di atas, beberapa hal yang perlu diperhatikan adalah:

- **Baseline XGBoost** menjadi pembanding utama karena model dilatih menggunakan target harga asli dan parameter dasar.
- **Label Price Logaritmik** mengurangi pengaruh harga rumah yang sangat tinggi dan memperbaiki RMSLE.
- **Tuned + Label Logaritmik** menggunakan regularisasi, subsampling, dan learning rate yang lebih kecil agar model lebih stabil.
- Perbedaan RMSE dan RMSLE perlu dibaca bersama karena skenario dengan RMSE terendah belum tentu memiliki RMSLE terendah.

### Kesimpulan Bagian XGBoost Regressor

Skenario dengan RMSLE terendah dipilih sebagai model XGBoost terbaik untuk prediksi data test. Untuk evaluasi yang lebih stabil, pure XGBoost juga diuji menggunakan 5-fold cross-validation dan memperoleh CV log RMSE sekitar `0.111-0.112`.


### Tahap Testing pada Data Test

Setelah skenario terbaik ditentukan dari validation set, konfigurasi tersebut digunakan kembali untuk melatih model pada seluruh data training. Model final kemudian digunakan untuk memprediksi `SalePrice` pada data test dan menghasilkan file submission.


In [ ]:
# Preprocessing seluruh data untuk training model final.
X_full_fe = engineer_features(X)
X_test_submission_fe = engineer_features(X_test_final)

final_preprocessor_xgb = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("ord", ordinal_transformer, ordinal_features),
        ("nom", nominal_transformer, onehot_features),
    ],
    remainder="drop",
)
X_full_prepared_xgb = final_preprocessor_xgb.fit_transform(X_full_fe)
X_test_prepared_xgb = final_preprocessor_xgb.transform(X_test_submission_fe)

xgb_final_model = clone(xgb_best_result["Model"])
xgb_final_target = (
    np.log1p(y)
    if xgb_best_result["Label Logaritmik"] == "Ya"
    else y
)
xgb_final_model.fit(X_full_prepared_xgb, xgb_final_target)
xgb_final_prediction = xgb_final_model.predict(X_test_prepared_xgb)
if xgb_best_result["Label Logaritmik"] == "Ya":
    xgb_final_prediction = np.expm1(xgb_final_prediction)
xgb_final_prediction = np.clip(xgb_final_prediction, 0, None)

xgb_submission = pd.DataFrame(
    {"Id": test_df["Id"], "SalePrice": xgb_final_prediction}
)
xgb_submission_path = OUTPUT_DIR / "submission_xgboost.csv"
xgb_submission.to_csv(xgb_submission_path, index=False)

print("Submission disimpan di:", xgb_submission_path)
xgb_submission.head()


## 5. Model - Random Forest Regressor (Rendy Tanuwijaya - 5025241099)

Bagian ini mengerjakan eksperimen model **Random Forest Regressor** untuk memprediksi `SalePrice`. Eksperimen dibuat untuk membandingkan performa model pada kondisi normal dan pada dua skenario uji coba:

1. **Feature Selection**: model menggunakan sejumlah fitur terbaik berdasarkan skor `mutual_info_regression`.
2. **Label Price Logaritmik**: target `SalePrice` ditransformasi dengan `log1p` saat training, lalu prediksi dikembalikan ke skala harga asli dengan `expm1`.

Eksperimen juga menyertakan kombinasi kedua skenario tersebut agar perbandingannya sejalan dengan model lainnya.

### Setup Eksperimen

Metrik evaluasi yang digunakan adalah:

* **RMSE**: menunjukkan rata-rata besar error dalam satuan harga rumah, dengan penalti lebih besar untuk error besar.
* **MAE**: menunjukkan rata-rata absolut error dalam satuan harga rumah.
* **RMSLE**: cocok untuk target harga karena mengukur error relatif pada skala logaritmik.
* **R2 Score**: menunjukkan proporsi variasi target yang mampu dijelaskan model.

Bagian ini juga mendefinisikan *grid hyperparameter* khusus untuk Random Forest Regressor yang mencakup iterasi kombinasi `n_estimators`, `max_depth`, `min_samples_split`, `min_samples_leaf`, dan `max_features`. Semua eksperimen menggunakan data training dan validation split yang sama dari tahap preprocessing.

In [ ]:
from sklearn.feature_selection import SelectKBest, mutual_info_regression
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_squared_log_error, r2_score
from sklearn.ensemble import RandomForestRegressor
import numpy as np

try:
    import matplotlib.pyplot as plt
    HAS_MATPLOTLIB = True
except ModuleNotFoundError:
    HAS_MATPLOTLIB = False
    print('matplotlib belum tersedia, sehingga cell grafik akan dilewati.')

RANDOM_STATE = 42

def evaluate_regression_rf(y_true, y_pred):
    """Menghitung metrik regresi pada skala harga asli."""
    y_pred_safe = np.clip(y_pred, 0, None)
    return {
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSLE': np.sqrt(mean_squared_log_error(y_true, y_pred_safe)),
        'R2': r2_score(y_true, y_pred)
    }

def get_candidate_k_values_rf(n_features):
    """Menentukan pilihan jumlah fitur untuk skenario feature selection."""
    base_values = [20, 40, 60, 80, 120, 160]
    k_values = sorted({k for k in base_values if k < n_features})
    k_values.append('all')
    return k_values

param_grid_rf = [
    {
        'n_estimators': n_estimators,
        'max_depth': max_depth,
        'min_samples_split': min_samples_split,
        'min_samples_leaf': min_samples_leaf,
        'max_features': max_features
    }
    for n_estimators in [50, 100]
    for max_depth in [10, 20, None]
    for min_samples_split in [2, 5]
    for min_samples_leaf in [1, 2]
    for max_features in ['sqrt', 1.0]
]

print('Jumlah kombinasi hyperparameter RF:', len(param_grid_rf))
print('Jumlah fitur setelah preprocessing:', X_train_prepared_df.shape[1])

### Fungsi Training dan Evaluasi

Fungsi berikut menjalankan satu eksperimen Random Forest Regressor. Jika `use_feature_selection=True`, jumlah fitur terbaik ikut dicari dari beberapa kandidat `k`. Jika `use_log_target=True`, model dilatih pada target `log1p(SalePrice)` dan hasil prediksi dikembalikan ke skala harga asli dengan `expm1` sebelum evaluasi. Proses *training* model ini memanfaatkan `n_jobs=-1` agar berjalan paralel secara maksimal.

In [ ]:
def run_random_forest_experiment(name, use_feature_selection=False, use_log_target=False):
    y_train_model = np.log1p(y_train) if use_log_target else y_train
    selection_target = y_train_model if use_log_target else y_train

    k_candidates = get_candidate_k_values_rf(X_train_prepared_df.shape[1]) if use_feature_selection else ['all']
    best_result = None

    for k in k_candidates:
        if use_feature_selection and k != 'all':
            selector = SelectKBest(
                score_func=lambda X_data, y_data: mutual_info_regression(
                    X_data,
                    y_data,
                    random_state=RANDOM_STATE
                ),
                k=k
            )
            X_train_model = selector.fit_transform(X_train_prepared_df, selection_target)
            X_valid_model = selector.transform(X_valid_prepared_df)
            selected_features = X_train_prepared_df.columns[selector.get_support()].tolist()
        else:
            selector = None
            X_train_model = X_train_prepared_df
            X_valid_model = X_valid_prepared_df
            selected_features = X_train_prepared_df.columns.tolist()

        for params in param_grid_rf:
            # n_jobs=-1 agar training Random Forest berjalan paralel secara maksimal
            model = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1, **params)
            model.fit(X_train_model, y_train_model)
            y_pred_model = model.predict(X_valid_model)
            y_pred = np.expm1(y_pred_model) if use_log_target else y_pred_model

            metrics = evaluate_regression_rf(y_valid, y_pred)
            result = {
                'Skenario': name,
                'Feature Selection': 'Ya' if use_feature_selection else 'Tidak',
                'Label Logaritmik': 'Ya' if use_log_target else 'Tidak',
                'Jumlah Fitur': len(selected_features),
                'RMSE': metrics['RMSE'],
                'MAE': metrics['MAE'],
                'RMSLE': metrics['RMSLE'],
                'R2': metrics['R2'],
                'Best Params': params,
                'Selected Features': selected_features,
                'Model': model,
                'Selector': selector,
                'Prediction': y_pred
            }

            if best_result is None or result['RMSE'] < best_result['RMSE']:
                best_result = result

    return best_result

### Menjalankan Seluruh Skenario Uji Coba

Empat eksperimen berikut digunakan agar dampak setiap skenario dapat dibandingkan dengan jelas:

* **Baseline Random Forest**: tanpa feature selection dan tanpa transformasi log target.
* **Feature Selection RF**: hanya menggunakan fitur terpilih.
* **Label Price Logaritmik RF**: target harga ditransformasi logaritmik.
* **Feature Selection + Label Logaritmik RF**: kombinasi kedua skenario.

In [ ]:
experiments_rf = [
    ('Baseline Random Forest', False, False),
    ('Feature Selection RF', True, False),
    ('Label Price Logaritmik RF', False, True),
    ('Feature Selection + Label Logaritmik RF', True, True)
]

experiment_results_rf = []

for experiment_name, use_feature_selection, use_log_target in experiments_rf:
    print(f'Menjalankan eksperimen: {experiment_name}')
    result = run_random_forest_experiment(
        experiment_name,
        use_feature_selection=use_feature_selection,
        use_log_target=use_log_target
    )
    experiment_results_rf.append(result)

print('Seluruh eksperimen Random Forest selesai.')

### Tabel Hasil Performa Tiap Skenario

Tabel berikut menunjukkan performa terbaik dari masing-masing skenario berdasarkan pencarian hyperparameter sederhana. Hasil evaluasi metrik dibulatkan menjadi 4 angka di belakang koma, dan model terbaik dipilih dan diurutkan berdasarkan nilai **RMSE** terendah pada validation set.

In [ ]:
results_rf_df = pd.DataFrame(experiment_results_rf)

performance_columns_rf = [
    'Skenario', 'Feature Selection', 'Label Logaritmik', 'Jumlah Fitur',
    'RMSE', 'MAE', 'RMSLE', 'R2', 'Best Params'
]

performance_table_rf = results_rf_df[performance_columns_rf].copy()
performance_table_rf[['RMSE', 'MAE', 'RMSLE', 'R2']] = performance_table_rf[['RMSE', 'MAE', 'RMSLE', 'R2']].round(4)
performance_table_rf = performance_table_rf.sort_values('RMSE').reset_index(drop=True)
performance_table_rf

## Visualisasi Hasil Eksperimen Random Forest

Bagian ini memvisualisasikan hasil eksperimen secara komprehensif untuk mempermudah analisis struktural model:

1. **Perbandingan 4 Metrik Utama (Subplots 2x2)**: Menampilkan perbandingan performa tiap skenario dari berbagai sudut pandang metrik (RMSE, MAE, RMSLE, dan R2). Visualisasi multimetrik ini krusial untuk memastikan apakah penurunan *error* absolut (RMSE/MAE) berjalan beriringan dengan peningkatan kecocokan rasio/proporsi data (R2/RMSLE).
2. **Feature Importances**: Mengekstraksi dan memvisualisasikan 15 fitur dengan tingkat kontribusi terbesar terhadap prediksi harga rumah. Nilai bobot ini diambil secara otomatis dari arsitektur model dengan performa paling optimal pada eksperimen di atas.

In [ ]:
if HAS_MATPLOTLIB:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Perbandingan 4 Metrik Evaluasi Random Forest', fontsize=16, y=1.02)

    scenarios = performance_table_rf['Skenario']
    metrics = [
        ('RMSE', axes[0, 0], '#1f77b4', 'Lower Better'),
        ('MAE', axes[0, 1], '#ff7f0e', 'Lower Better'),
        ('RMSLE', axes[1, 0], '#2ca02c', 'Lower Better'),
        ('R2', axes[1, 1], '#d62728', 'Higher Better')
    ]

    for metric_name, ax, color, description in metrics:
        scores = performance_table_rf[metric_name]
        bars = ax.bar(scenarios, scores, color=color)
        ax.set_title(f'{metric_name} ({description})', fontsize=12)
        ax.set_xticklabels(scenarios, rotation=25, ha='right')

        for bar in bars:
            yval = bar.get_height()
            text_format = f'{yval:,.4f}' if metric_name in ['RMSLE', 'R2'] else f'{yval:,.0f}'

            y_pos = yval + (yval * 0.01) if metric_name != 'R2' else yval - (yval * 0.05)
            va_align = 'bottom' if metric_name != 'R2' else 'top'

            ax.text(bar.get_x() + bar.get_width()/2, y_pos, text_format,
                    ha='center', va=va_align, fontsize=9, fontweight='bold')

    plt.tight_layout()
    plt.show()

    best_scenario = performance_table_rf.iloc[0]['Skenario']
    best_result = next(res for res in experiment_results_rf if res['Skenario'] == best_scenario)

    best_model = best_result['Model']
    feature_names = best_result['Selected Features']
    importances = best_model.feature_importances_

    top_n = min(15, len(feature_names))
    indices = np.argsort(importances)[::-1][:top_n]

    plt.figure(figsize=(10, 8))
    plt.title(f'Top {top_n} Feature Importances\n(Model Terbaik: {best_scenario})', fontsize=14, pad=15)
    plt.barh(range(top_n), importances[indices], align='center', color='skyblue', edgecolor='black')
    plt.yticks(range(top_n), [feature_names[i] for i in indices])
    plt.xlabel('Tingkat Kepentingan Relatif (Relative Importance)', fontsize=12)
    plt.gca().invert_yaxis()

    plt.tight_layout()
    plt.show()

## Kesimpulan Eksperimen Random Forest

Dari serangkaian pengujian di atas, kita dapat menarik beberapa kesimpulan mengenai perilaku fundamental arsitektur Random Forest pada dataset *Ames House Prices*:

1. **Dampak Transformasi Logaritmik pada Target**:
   Fungsi *loss* bawaan pada algoritma berbasis *tree* (seperti penghitungan *Mean Squared Error* pada tiap *split* daun) sangat sensitif terhadap nilai ekstrem. Karena distribusi harga properti dunia nyata cenderung *right-skewed* (condong ke beberapa rumah yang bernilai sangat mahal), tanpa transformasi, pohon-pohon akan terlalu banyak beradaptasi untuk menekan *error* pada *outlier* tersebut. Transformasi `log1p` mengompresi variansi ekstrem ini, memungkinkan struktur *tree* membagi data (*splitting*) dengan jauh lebih seimbang untuk merepresentasikan harga mayoritas rumah. Hal ini biasanya ditandai dengan perbaikan yang sangat signifikan pada nilai R2 dan RMSLE.

2. **Peran Feature Selection vs. Mekanisme Bawaan Model**:
   Berbeda dengan model linear murni, Random Forest secara inheren sudah memiliki mekanisme pemilahan fitur saat membandingkan *split* terbaik pada setiap node (terutama karena parameter `max_features`). Akibatnya, pre-seleksi fitur (menggunakan *Mutual Information*) mungkin tidak selalu memberikan lonjakan metrik performa yang masif. Namun secara sistematis, memangkas fitur-fitur bising (*noise*) di awal terbukti membuat model lebih efisien dalam melakukan komputasi ratusan pohon, serta meminimalisasi risiko *overfitting* pada fitur yang tidak relevan.

3. **Validasi Feature Importances**:
   Distribusi bobot fitur dari model terbaik berhasil memvalidasi logika penentuan harga properti. Fitur struktural utama seperti kualitas bangunan keseluruhan (*OverallQual*), luas area hidup (*GrLivArea*), atau kapasitas garasi tetap mendominasi puncak *importances*, membuktikan bahwa model mampu mengidentifikasi pola fundamental arsitektur rumah alih-alih sekadar menghafal data *training*.